In [ ]:
import numpy as np
from astropy.table import Table
from mast_aladin import MastAladin
from sidecar import Sidecar
%load_ext memory_profiler

In [ ]:
def generate_quadrilateral_sregion(ra_center, dec_center, size_arcsec=120):
    size_deg = size_arcsec / 3600.0
    half_size = size_deg / 2
    ra_scale = np.cos(np.radians(dec_center))
    
    ra1, dec1 = ra_center - half_size/ra_scale, dec_center - half_size
    ra2, dec2 = ra_center - half_size/ra_scale, dec_center + half_size
    ra3, dec3 = ra_center + half_size/ra_scale, dec_center + half_size
    ra4, dec4 = ra_center + half_size/ra_scale, dec_center - half_size
    return f"POLYGON ICRS {ra1:.6f} {dec1:.6f} {ra2:.6f} {dec2:.6f} {ra3:.6f} {dec3:.6f} {ra4:.6f} {dec4:.6f}"

def generate_circle_sregion(ra_center, dec_center, radius_arcsec=60):
    radius_deg = radius_arcsec / 3600.0
    return f"CIRCLE ICRS {ra_center:.6f} {dec_center:.6f} {radius_deg:.6f}"

def generate_polygon_sregion(ra_center, dec_center, n_points=16, radius_arcsec=60):
    radius_deg = radius_arcsec / 3600.0
    angles = np.linspace(0, 2*np.pi, n_points, endpoint=False)
    coords = []
    ra_scale = np.cos(np.radians(dec_center))
    for angle in angles:
        ra = ra_center + (radius_deg * np.cos(angle)) / ra_scale
        dec = dec_center + radius_deg * np.sin(angle)
        coords.append(f"{ra:.6f} {dec:.6f}")
    return f"POLYGON ICRS {' '.join(coords)}"

def create_table_with_sregion(n_sources, sregion_type='quadrilateral', ra_range=(10, 11), dec_range=(20, 21)):
    np.random.seed(42)
    ra = np.random.uniform(ra_range[0], ra_range[1], n_sources)
    dec = np.random.uniform(dec_range[0], dec_range[1], n_sources)
    
    sregions = []
    for i in range(n_sources):
        if sregion_type == 'quadrilateral':
            sregions.append(generate_quadrilateral_sregion(ra[i], dec[i]))
        elif sregion_type == 'circle':
            sregions.append(generate_circle_sregion(ra[i], dec[i]))
        elif sregion_type == 'polygon16':
            sregions.append(generate_polygon_sregion(ra[i], dec[i], n_points=16))
    
    return Table({'s_region': sregions, 'source_id': np.arange(n_sources)})

In [ ]:
aladin = MastAladin(target="10.5 20.5", fov=2)
aladin

In [ ]:
n_sources = 200000
sregion_type = 'polygon16'

table = create_table_with_sregion(n_sources, sregion_type)

In [ ]:
%%time
%memit

aladin.add_graphic_overlay_from_stcs(table['s_region'])

In [ ]:
table_with_coords = table.copy()
np.random.seed(42)
table_with_coords['ra'] = np.random.uniform(10, 11, n_sources)
table_with_coords['dec'] = np.random.uniform(20, 21, n_sources)

In [ ]:
%%time
%memit

aladin.add_table(table_with_coords)